# 02 — Preprocessing
**Dataset:** Air Quality Data in India (2015–2020) — `city_day.csv`

This notebook takes the raw data and produces a fully cleaned checkpoint:
1. Load raw data
2. Inspect missing values
3. Drop unusable columns
4. Handle missing values (interpolation + median fill)
5. Drop rows with missing target
6. Remove duplicates
7. Cap outliers
8. Save the cleaned checkpoint to `data/interim/`

## 1. Imports

In [4]:
import pandas as pd
import numpy as np


## 2. Load raw data
Always read from `data/raw/` — never edit this file directly.

In [17]:
df = pd.read_csv(r"f:\programs\ML-basic\Clean-Air-Climate-Resilience\data\raw\city_day.csv")

df['Date'] = pd.to_datetime(df['Date'])
df.shape


(29531, 16)

## 3. Inspect missing values
Check counts and percentages before deciding what to do with each column.

In [6]:
df.isnull().sum()

City              0
Date              0
PM2.5          4598
PM10          11140
NO             3582
NO2            3585
NOx            4185
NH3           10328
CO             2059
SO2            3854
O3             4022
Benzene        5623
Toluene        8041
Xylene        18109
AQI            4681
AQI_Bucket     4681
dtype: int64

In [7]:
missing_pct = df.isnull().sum() / len(df) * 100
missing_pct.sort_values(ascending=False)


Xylene        61.322001
PM10          37.723071
NH3           34.973418
Toluene       27.229014
Benzene       19.041008
AQI           15.851139
AQI_Bucket    15.851139
PM2.5         15.570079
NOx           14.171549
O3            13.619586
SO2           13.050692
NO2           12.139785
NO            12.129626
CO             6.972334
City           0.000000
Date           0.000000
dtype: float64

## 4. Drop columns that are too sparse to impute reliably
`Xylene` is typically ~60%+ missing — not reliably fixable, so it's dropped entirely.

In [8]:
df = df.drop(columns=['Xylene'])
df.shape


(29531, 15)

## 5. Handle missing values in pollutant columns
Sort by City + Date, then interpolate per city (time-based) rather than using a global mean/median, since pollution levels differ a lot city to city.

In [9]:
df = df.sort_values(['City', 'Date'])

pollutant_cols = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene']

df[pollutant_cols] = df.groupby('City')[pollutant_cols].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)


Interpolation can't fill values at the very start/end of a city's series (no neighboring point on one side) — fill any leftovers with that city's median.

In [10]:
for col in pollutant_cols:
    df[col] = df[col].fillna(df.groupby('City')[col].transform('median'))

df[pollutant_cols].isnull().sum()  # should now all be 0


PM2.5         0
PM10       2009
NO            0
NO2           0
NOx        1169
NH3        2009
CO            0
SO2           0
O3          162
Benzene    2732
Toluene    4010
dtype: int64

## 6. Drop rows where the target (AQI / AQI_Bucket) is missing
The target should never be imputed/faked — rows without it aren't usable for training.

In [11]:
before = len(df)
df = df.dropna(subset=['AQI', 'AQI_Bucket'])
after = len(df)
print(f"Dropped {before - after} rows ({(before-after)/before*100:.2f}%) missing AQI/AQI_Bucket")


Dropped 4681 rows (15.85%) missing AQI/AQI_Bucket


In [12]:
df.isnull().sum()  # final check — should be all zeros now

City             0
Date             0
PM2.5            0
PM10          1893
NO               0
NO2              0
NOx            771
NH3           1334
CO               0
SO2              0
O3             153
Benzene       2259
Toluene       3309
AQI              0
AQI_Bucket       0
dtype: int64

## 7. Remove duplicate records

In [13]:
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate (City, Date) pairs:", df.duplicated(subset=['City','Date']).sum())

df = df.drop_duplicates()
df = df.drop_duplicates(subset=['City','Date'], keep='first')
df.shape


Full-row duplicates: 0
Duplicate (City, Date) pairs: 0


(24850, 15)

## 8. Cap outliers (IQR method)
Sensor faults occasionally produce extreme spikes. Capping (not deleting) preserves genuinely high-pollution days while limiting the effect of clearly erroneous readings.

In [14]:
def cap_outliers_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return series.clip(lower, upper)

for col in pollutant_cols + ['AQI']:
    df[col] = cap_outliers_iqr(df[col])

df[pollutant_cols + ['AQI']].describe().T


,count,mean,std,min,25%,50%,75%,max
PM2.5,24850.0,62.589650,44.619934,0.04,28.9800,48.93,82.3500,162.40500
PM10,22957.0,118.355048,77.911700,0.03,54.5200,100.51,158.0800,313.42000
NO,24850.0,14.731958,12.507944,0.03,5.6625,9.98,20.2400,42.10625
NO2,24850.0,27.542067,19.925050,0.01,11.8100,21.95,38.0500,77.41000
NOx,24079.0,29.551358,22.083383,0.00,13.4600,23.63,40.0100,79.83500
NH3,23516.0,25.000592,20.058056,0.01,8.9100,18.53,39.6900,85.86000
CO,24850.0,1.131985,0.780268,0.00,0.5800,0.94,1.4800,2.83000
SO2,24850.0,11.809863,8.255561,0.01,5.7100,9.23,15.3275,29.75375
O3,24697.0,34.442547,19.974961,0.01,19.3300,31.27,46.1300,86.33000
Benzene,22591.0,2.237767,2.488707,0.00,0.2400,1.30,3.4100,8.16500


## 9. Final sanity checks

In [15]:
print("Shape:", df.shape)
print("\nRemaining nulls:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T


Shape: (24850, 15)

Remaining nulls:
 City             0
Date             0
PM2.5            0
PM10          1893
NO               0
NO2              0
NOx            771
NH3           1334
CO               0
SO2              0
O3             153
Benzene       2259
Toluene       3309
AQI              0
AQI_Bucket       0
dtype: int64

Duplicate rows: 0


,count,mean,min,25%,50%,75%,max,std
Date,24850,2018-07-24 18:51:25.714285568,2015-01-01 00:00:00,2017-08-16 00:00:00,2018-11-05 00:00:00,2019-10-11 00:00:00,2020-07-01 00:00:00,NaN
PM2.5,24850.0,62.58965,0.04,28.98,48.93,82.35,162.405,44.619934
PM10,22957.0,118.355048,0.03,54.52,100.51,158.08,313.42,77.9117
NO,24850.0,14.731958,0.03,5.6625,9.98,20.24,42.10625,12.507944
NO2,24850.0,27.542067,0.01,11.81,21.95,38.05,77.41,19.92505
NOx,24079.0,29.551358,0.0,13.46,23.63,40.01,79.835,22.083383
NH3,23516.0,25.000592,0.01,8.91,18.53,39.69,85.86,20.058056
CO,24850.0,1.131985,0.0,0.58,0.94,1.48,2.83,0.780268
SO2,24850.0,11.809863,0.01,5.71,9.23,15.3275,29.75375,8.255561
O3,24697.0,34.442547,0.01,19.33,31.27,46.13,86.33,19.974961


## 10. Save the cleaned checkpoint
Saved to `data/interim/` — the next notebook (`03_feature_engineering.ipynb`) will load from here, not from `raw/`.

In [18]:
df.to_csv("../data/interim/city_day_nulls_handled.csv", index=False)
print("Saved:", df.shape)


Saved: (29531, 16)
